# Практическая работа: Масштабирование и кодирование признаков
**Студент:** Шетинин М.Ю., группа ИУ5-64Б

**Набор данных:** Graduate Admissions (Admission_Predict_Ver1.1.csv)

**Задачи:**
1. Масштабирование данных (для одного признака)
2. Преобразование категориальных признаков двумя способами (Label Encoding, One-Hot Encoding)
3. Violin plot для произвольной колонки

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

import warnings
warnings.filterwarnings('ignore')

print('Библиотеки загружены успешно')

## 1. Загрузка данных

In [ ]:
df = pd.read_csv('Admission_Predict_Ver1.1.csv')
print('Форма датасета:', df.shape)
df.head()

In [ ]:
print('Информация о датасете:')
df.info()
print('\nСтатистика:')
df.describe()

## 2. Создание категориального признака

**Примечание:** В датасете Graduate Admissions отсутствуют категориальные признаки. Согласно условию задания, создадим категориальный признак искусственно на основе числового признака `CGPA` (средний балл).

In [ ]:
# Создаём категориальный признак 'GPA_Category' на основе CGPA
def categorize_gpa(gpa):
    if gpa < 7.0:
        return 'Low'
    elif gpa < 8.5:
        return 'Medium'
    else:
        return 'High'

df['GPA_Category'] = df['CGPA'].apply(categorize_gpa)

print('Распределение категорий GPA_Category:')
print(df['GPA_Category'].value_counts())
df[['CGPA', 'GPA_Category']].head(10)

## 3. Масштабирование данных

**Метод:** `MinMaxScaler` из `sklearn.preprocessing`

**Почему:** MinMaxScaler приводит значения признака к диапазону [0, 1], что полезно для алгоритмов, чувствительных к масштабу (например, KNN, SVM, нейронные сети). Признак `GRE Score` имеет диапазон 260–340, что может доминировать над другими признаками.

In [ ]:
scaler = MinMaxScaler()
df['GRE Score Scaled'] = scaler.fit_transform(df[['GRE Score']])

print('GRE Score до масштабирования:')
print(f'  Min: {df["GRE Score"].min()}, Max: {df["GRE Score"].max()}')
print(f'  Mean: {df["GRE Score"].mean():.2f}, Std: {df["GRE Score"].std():.2f}')

print('\nGRE Score после масштабирования (MinMaxScaler):')
print(f'  Min: {df["GRE Score Scaled"].min():.4f}, Max: {df["GRE Score Scaled"].max():.4f}')
print(f'  Mean: {df["GRE Score Scaled"].mean():.4f}, Std: {df["GRE Score Scaled"].std():.4f}')

df[['GRE Score', 'GRE Score Scaled']].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['GRE Score'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('GRE Score (оригинал)')
axes[0].set_xlabel('GRE Score')
axes[0].set_ylabel('Частота')

axes[1].hist(df['GRE Score Scaled'], bins=20, color='darkorange', edgecolor='white')
axes[1].set_title('GRE Score после MinMaxScaler')
axes[1].set_xlabel('GRE Score Scaled')
axes[1].set_ylabel('Частота')

plt.tight_layout()
plt.savefig('scaling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('График сохранён: scaling_comparison.png')

## 4. Label Encoding

**Метод:** `LabelEncoder` из `sklearn.preprocessing`

**Почему:** Label Encoding заменяет каждую категорию целым числом. Подходит для древовидных моделей (Decision Tree, Random Forest), которые не предполагают порядка между числами. Для признака `GPA_Category` можно использовать этот метод, однако важно учитывать, что линейные модели могут некорректно интерпретировать полученные числа как порядковые.

In [ ]:
le = LabelEncoder()
df['GPA_Category_LE'] = le.fit_transform(df['GPA_Category'])

print('Маппинг Label Encoding:')
for cls, code in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {cls} -> {code}')

df[['GPA_Category', 'GPA_Category_LE']].head(10)

## 5. One-Hot Encoding

**Метод:** `pd.get_dummies()` из библиотеки pandas

**Почему:** One-Hot Encoding создаёт отдельный бинарный столбец для каждой категории, устраняя ложный порядок между значениями. Подходит для линейных моделей и нейронных сетей. Недостаток — увеличение размерности при большом числе уникальных категорий («проклятие размерности»).

In [ ]:
ohe_columns = pd.get_dummies(df['GPA_Category'], prefix='GPA')
df = pd.concat([df, ohe_columns], axis=1)

print('Новые столбцы после One-Hot Encoding:')
print(ohe_columns.columns.tolist())

df[['GPA_Category', 'GPA_High', 'GPA_Low', 'GPA_Medium']].head(10)

## 6. Violin Plot

Скрипичная диаграмма (violin plot) сочетает в себе boxplot и оценку плотности распределения. Построим её для признака `CGPA` в разрезе категорий `Research` (наличие/отсутствие опыта исследований).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Violin plot 1: CGPA по наличию исследовательского опыта
df_plot = df.copy()
df_plot['Research'] = df_plot['Research'].map({0: 'Нет опыта', 1: 'Есть опыт'})

sns.violinplot(
    data=df_plot,
    x='Research',
    y='CGPA',
    palette='Set2',
    inner='box',
    ax=axes[0]
)
axes[0].set_title('Violin Plot: CGPA по исследовательскому опыту', fontsize=13)
axes[0].set_xlabel('Исследовательский опыт')
axes[0].set_ylabel('CGPA')

# Violin plot 2: GRE Score по категории GPA
category_order = ['Low', 'Medium', 'High']
sns.violinplot(
    data=df,
    x='GPA_Category',
    y='GRE Score',
    order=category_order,
    palette='muted',
    inner='quartile',
    ax=axes[1]
)
axes[1].set_title('Violin Plot: GRE Score по категории GPA', fontsize=13)
axes[1].set_xlabel('Категория GPA')
axes[1].set_ylabel('GRE Score')

plt.tight_layout()
plt.savefig('violin_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('График сохранён: violin_plots.png')

## 7. Итоговый датафрейм

In [ ]:
result_cols = [
    'Serial No.', 'GRE Score', 'GRE Score Scaled',
    'CGPA', 'GPA_Category', 'GPA_Category_LE',
    'GPA_High', 'GPA_Low', 'GPA_Medium'
]
print('Итоговый датафрейм (первые 10 строк):')
df[result_cols].head(10)

In [ ]:
df.to_csv('Admission_Predict_Processed.csv', index=False)
print('Обработанный датасет сохранён: Admission_Predict_Processed.csv')

## 8. Выводы

### Методы и обоснование:

**Масштабирование — MinMaxScaler:**
- Приводит значения к диапазону [0, 1] по формуле: `x_scaled = (x - x_min) / (x_max - x_min)`
- Применён к признаку `GRE Score` (диапазон 260–340)
- Выбран, т.к. данные не имеют явных выбросов, а многие ML-алгоритмы (KNN, SVM, нейронные сети) чувствительны к масштабу признаков

**Label Encoding:**
- Каждой категории присваивается целое число: High→0, Low→1, Medium→2
- Подходит для древовидных моделей (Decision Tree, Random Forest, Gradient Boosting)
- Компактный: не увеличивает размерность
- Недостаток: линейные модели могут интерпретировать числа как порядок

**One-Hot Encoding:**
- Создаёт бинарные столбцы для каждой категории (GPA_High, GPA_Low, GPA_Medium)
- Устраняет ложный порядок между категориями
- Подходит для линейных моделей и нейронных сетей
- Недостаток: увеличивает размерность данных

**Violin Plot:**
- Сочетает boxplot (медиана, квартили) и KDE (плотность распределения)
- Наглядно показывает, что студенты с исследовательским опытом имеют более высокий CGPA и более сосредоточенное распределение
- Категория High GPA ожидаемо коррелирует с более высокими баллами GRE